In [7]:
import numpy as np
import pickle
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from collections import Counter
from sklearn.utils import resample

def print_class_distribution(labels, label_encoder, title):
    """Print the number of samples for each class."""
    label_counts = Counter(labels)
    print(f"{title}:")
    for class_index, count in label_counts.items():
        class_label = label_encoder.inverse_transform([class_index])[0]
        print(f"  Class '{class_label}': {count} samples")

def run_classification_task(pickle_file_path):

    # Load the data from the pickle file
    with open(pickle_file_path, 'rb') as handle:
        data = pickle.load(handle)

    # Ensure the data is a numpy array
    if not isinstance(data, dict) or 'embeddings' not in data or 'labels' not in data:
        raise ValueError("The pickle file does not contain the expected 'embeddings' and 'labels' keys.")
    
    embeddings = np.array(data['embeddings'])
    labels = np.array(data['labels'])

    # Encode the labels as integers
    label_encoder = LabelEncoder()
    try:
        labels_encoded = label_encoder.fit_transform(labels)
    except Exception as e:
        print(f"Error encoding labels: {e}")
        print(f"First 10 labels: {labels[:10]}")
        return

    # Print the shape and type of the resampled numpy arrays
    print(f"Shape of the embeddings array: {embeddings.shape}")
    print(f"Shape of the labels array: {labels.shape}")

    # Print class distribution before balancing
    print_class_distribution(labels_encoded, label_encoder, "Class distribution before balancing")

    # Balance the dataset by undersampling
    min_class_size = min(Counter(labels_encoded).values())
    print(f"Smallest class size: {min_class_size}")

    balanced_embeddings = []
    balanced_labels = []

    for class_index in np.unique(labels_encoded):
        class_mask = labels_encoded == class_index
        class_embeddings = embeddings[class_mask]
        class_labels = labels_encoded[class_mask]
        
        # Undersample the class to the size of the smallest class
        resampled_embeddings, resampled_labels = resample(
            class_embeddings, class_labels, 
            replace=False,  # do not sample with replacement
            n_samples=min_class_size, 
            random_state=42
        )

        balanced_embeddings.append(resampled_embeddings)
        balanced_labels.append(resampled_labels)

    balanced_embeddings = np.vstack(balanced_embeddings)
    balanced_labels = np.hstack(balanced_labels)

    # Print class distribution after balancing
    print_class_distribution(balanced_labels, label_encoder, "Class distribution after balancing")

    # Split the balanced data into training and testing sets using StratifiedShuffleSplit
    X_train, X_test, y_train, y_test = train_test_split(
        balanced_embeddings, balanced_labels, 
        test_size=0.2, 
        stratify=balanced_labels, 
        random_state=42
    )

    # Print class distribution after split
    print_class_distribution(y_train, label_encoder, "Class distribution in training set")
    print_class_distribution(y_test, label_encoder, "Class distribution in testing set")

    # Apply PCA to keep 99% of the variance
    pca = PCA(n_components=0.99)
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)

    # Print the number of features kept after PCA
    n_features_kept = X_train_pca.shape[1]
    print(f"Number of features kept after PCA: {n_features_kept}")

    # Define the parameter grid for the RandomizedSearchCV
    random_grid_knn = {
        "n_neighbors": [3, 5, 7, 9, 11],
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan", "minkowski"]
    }

    # Set up the RandomizedSearchCV
    search = RandomizedSearchCV(
        estimator=KNeighborsClassifier(),
        param_distributions=random_grid_knn,
        scoring="balanced_accuracy",
        n_iter=50,
        cv=3,
        verbose=1,
        random_state=42,
        n_jobs=-1
    )

    # Fit the model
    search.fit(X_train_pca, y_train)

    # Get the best estimator
    clf = search.best_estimator_

    # Predict on the test set with a progress bar
    y_pred = []
    for i in tqdm(range(len(X_test_pca)), desc="Predicting"):
        y_pred.append(clf.predict(X_test_pca[i].reshape(1, -1)))

    y_pred = np.array(y_pred).flatten()

    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy}")

    # Print classification report
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

    # Plot confusion matrix
    conf_matrix = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(10, 7))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.title('Confusion Matrix')
    plt.show()

In [ ]:
pickle_file_dino_crc = "embeddings_patch_library/dinocrc100k.pkl"

# tranfer learning from our dataset to crc dataset

print("---- start classification for vit extracted features ---- ")
run_classification_task(pickle_file_dino_crc)

In [ ]:
pickle_file_resnet50_crc = "embeddings_patch_library/resnet50crc100k.pkl"

print("---- start classification for vit extracted features ---- ")
run_classification_task(pickle_file_resnet50_crc)